In [ ]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import pyulog
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R_
from pid_optimizer.plant_model import PlantModel
from pid_optimizer.gains import Gains, HERMIT_REFERENCE_GAINS
from pid_optimizer.pid_simulator import PX4Simulator, compute_rmse

LOG_PATH = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"
PLANT_PATH = "../artifacts/plant_model.json"

plant = PlantModel.load(PLANT_PATH)
gains = HERMIT_REFERENCE_GAINS
log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0 = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df.sort_values("t_s").reset_index(drop=True)

print("Plant:", plant)
print("Log loaded. Topics:", len(log.data_list))

In [ ]:
# ---------------------------------------------------------------------------
# Build setpoints DataFrame from ULG log.
#
# Topics available in this log:
#   vehicle_local_position_setpoint  — x, y, z, vx, vy, vz (885 msgs)
#   vehicle_attitude_setpoint        — roll_body, pitch_body, yaw_body (8738 msgs)
#   vehicle_local_position           — x, y, z (real state, 884 msgs)
#   vehicle_attitude                 — q[0..3] (real state, 17481 msgs)
# ---------------------------------------------------------------------------

DT = 0.004
T_MAX = 30.0

# --- Load raw DataFrames ---
pos_sp_df = to_df(log, "vehicle_local_position_setpoint")
att_sp_df = to_df(log, "vehicle_attitude_setpoint")
pos_df    = to_df(log, "vehicle_local_position")
att_df    = to_df(log, "vehicle_attitude")

# Clamp to T_MAX
pos_sp_df = pos_sp_df[pos_sp_df["t_s"] <= T_MAX].copy()
att_sp_df = att_sp_df[att_sp_df["t_s"] <= T_MAX].copy()
pos_df    = pos_df[pos_df["t_s"] <= T_MAX].copy()
att_df    = att_df[att_df["t_s"] <= T_MAX].copy()

print(f"pos_sp_df: {len(pos_sp_df)} rows, t=[{pos_sp_df['t_s'].min():.2f}, {pos_sp_df['t_s'].max():.2f}]")
print(f"att_sp_df: {len(att_sp_df)} rows, t=[{att_sp_df['t_s'].min():.2f}, {att_sp_df['t_s'].max():.2f}]")
print(f"pos_df:    {len(pos_df)} rows")
print(f"att_df:    {len(att_df)} rows")

# --- Build uniform time axis ---
t_start = max(pos_sp_df["t_s"].min(), att_sp_df["t_s"].min())
t_end   = min(pos_sp_df["t_s"].max(), att_sp_df["t_s"].max(), T_MAX)
t_sim   = np.arange(t_start, t_end, DT)
print(f"Simulation time: [{t_start:.2f}, {t_end:.2f}] s  ({len(t_sim)} steps @ DT={DT}s)")

# --- Helper: interpolate a column of a DataFrame onto t_sim ---
def interp_col(df: pd.DataFrame, col: str, t_out: np.ndarray) -> np.ndarray:
    return np.interp(t_out, df["t_s"].values, df[col].values)

# --- Interpolate position setpoints ---
sp_x  = interp_col(pos_sp_df, "x",  t_sim)
sp_y  = interp_col(pos_sp_df, "y",  t_sim)
sp_z  = interp_col(pos_sp_df, "z",  t_sim)
sp_vx = interp_col(pos_sp_df, "vx", t_sim)
sp_vy = interp_col(pos_sp_df, "vy", t_sim)
sp_vz = interp_col(pos_sp_df, "vz", t_sim)

# --- Interpolate attitude setpoints (from attitude_setpoint topic) ---
sp_roll  = interp_col(att_sp_df, "roll_body",  t_sim)
sp_pitch = interp_col(att_sp_df, "pitch_body", t_sim)
sp_yaw   = interp_col(att_sp_df, "yaw_body",   t_sim)

# --- Assemble setpoints DataFrame ---
setpoints = pd.DataFrame({
    "x":     sp_x,
    "y":     sp_y,
    "z":     sp_z,
    "vx":    sp_vx,
    "vy":    sp_vy,
    "vz":    sp_vz,
    "roll":  sp_roll,
    "pitch": sp_pitch,
    "yaw":   sp_yaw,
})

print("setpoints shape:", setpoints.shape)
print(setpoints.describe().round(3))

In [ ]:
sim = PX4Simulator(plant, gains)
result = sim.run(setpoints, dt=DT)
print("Simulation done. result shape:", result.shape)

# Real attitude: quaternion → Euler XYZ
q_real = att_df[["q[0]","q[1]","q[2]","q[3]"]].values  # w,x,y,z
euler_real = R_.from_quat(q_real[:, [1,2,3,0]]).as_euler("xyz")
t_real = att_df["t_s"].values

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
t_plot = t_sim[:len(result)]

# Position
for i, (col, label) in enumerate([("x","X(m)"), ("y","Y(m)"), ("z","Z(m)")]):
    ax = axes[i, 0]
    ax.plot(t_plot, result[col].values, label="simulated")
    ax.plot(pos_df["t_s"].values, pos_df[col].values, label="real", alpha=0.6)
    ax.set_ylabel(label); ax.legend(); ax.set_title(f"Position {label}")

# Attitude
for i, (idx, label) in enumerate([(0,"Roll(deg)"), (1,"Pitch(deg)"), (2,"Yaw(deg)")]):
    ax = axes[i, 1]
    ax.plot(t_plot, np.rad2deg(result[["roll","pitch","yaw"]].values[:, idx]), label="simulated")
    mask = t_real < T_MAX
    ax.plot(t_real[mask], np.rad2deg(euler_real[mask, idx]), label="real", alpha=0.6)
    ax.set_ylabel(label); ax.legend(); ax.set_title(f"Attitude {label}")

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
plt.suptitle("Simulator validation: simulated vs real state", fontsize=13)
plt.tight_layout(); plt.savefig("/tmp/sim_validation.png"); plt.close()
print("Saved to /tmp/sim_validation.png")

In [ ]:
ref = pd.DataFrame({
    "x": interp_col(pos_df, "x", t_sim),
    "y": interp_col(pos_df, "y", t_sim),
    "z": interp_col(pos_df, "z", t_sim),
})
rmse = compute_rmse(result[["x","y","z"]], ref[["x","y","z"]], cols=["x","y","z"])
print("Position RMSE:")
for col, val in rmse.items():
    sig_range = ref[col].max() - ref[col].min()
    pct = 100*val/sig_range if sig_range > 0.1 else float('nan')
    status = "OK" if pct < 15 else "WARN"
    print(f"  {col}: {val:.3f} m  ({pct:.1f}% of range)  [{status}]")